# Manual Build + Validate (no LLM)

This notebook lets you provide a custom `docker_build_pkg.sh` script (building_data) and validate it end-to-end against a repo commit using the datasmith Docker pipeline.

Flow:
- Edit the parameters below (owner/repo/sha, optional python version, output paths).
- Paste your build script into `building_data`.
- Run the Build + Validate cell.
- On success, the context is registered and saved to the chosen context registry JSON.

In [1]:
%load_ext autoreload
%autoreload 2
%cd /mnt/sdd1/atharvas/formulacode/datasmith/
from __future__ import annotations

import sys
import uuid
from pathlib import Path

# Ensure 'src' is importable (run this notebook from repo root)
SRC = (Path.cwd() / "src").resolve()
if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

import argparse

import docker
import pandas as pd

from datasmith.agents.build import _handle_success, build_once_with_context
from datasmith.core.models import Task
from datasmith.docker.cleanup import remove_containers_by_label
from datasmith.docker.context import ContextRegistry, DockerContext
from datasmith.docker.orchestrator import gen_run_labels
from datasmith.docker.validation import DockerValidator, ValidationConfig
from datasmith.logging_config import configure_logging

configure_logging(level=20)

print("Imports OK")

/mnt/sdd1/atharvas/formulacode/datasmith


21:55:17 WARNING  simple_useragent.core: Falling back to historic user agent.


Imports OK


## Parameters

Fill these in for your repo/commit, context-registry file, and output directory.


In [3]:
COMMITS_PATH = Path("scratch/artifacts/pipeflush/merge_commits_filtered_with_patch.parquet")
CONTEXT_REGISTRY_PATH = Path("scratch/merged_context_registry_2025-11-05T21:47:38.069445.json")
OUTPUT_DIR = Path("scratch/artifacts/pipeflush/results_synthesis_manual/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Prepare Docker + Registry

In [4]:
client = docker.from_env()
# Load or create a context registry
if CONTEXT_REGISTRY_PATH.exists():
    registry = ContextRegistry.load_from_file(CONTEXT_REGISTRY_PATH)
    print("Loaded registry from", CONTEXT_REGISTRY_PATH)
else:
    registry = ContextRegistry()
    print("Created new registry")

# Validation config (adjust timeouts if needed)
config = ValidationConfig(
    output_dir=OUTPUT_DIR,
    build_timeout=45 * 60,
    run_timeout=20 * 60,
    tail_chars=10_000,
)
validator = DockerValidator(client=client, context_registry=registry, machine_defaults={}, config=config)
print("Docker + registry ready")

Loaded registry from scratch/merged_context_registry_2025-11-05T21:47:38.069445.json
Docker + registry ready


In [5]:
commits_df = pd.read_parquet(COMMITS_PATH)
commits_df

,sha,date,message,total_additions,total_deletions,total_files_changed,files_changed,original_patch,has_asv,file_change_summary,...,pr_base_trees_url,pr_base_updated_at,pr_base_url,pr_base_visibility,pr_base_watchers,pr_base_watchers_count,pr_base_web_commit_signoff_required,pr_base_sha,container_name,patch
0,01fbbe37b2754f056b5241deef5f987482dc897e,2020-07-09T21:55:34+02:00,Memory leak testing using valgrind (#159)\n\n,49,0,3,.dockerignore\nREADME.rst\ndocker/Dockerfile.v...,From 01fbbe37b2754f056b5241deef5f987482dc897e ...,True,| File | Lines Added | Lines Removed | Total C...,...,https://api.github.com/repos/pygeos/pygeos/git...,2025-09-18T06:57:08Z,https://api.github.com/repos/pygeos/pygeos,public,388,388,False,d51e87ec1bd230bffb05882b3bf84b52540a89d1,pygeos-pygeos-d51e87ec1bd230bffb05882b3bf84b52...,diff --git a/.dockerignore b/.dockerignore\nne...
1,c05704f69fd0b9ec57fe4b31833736c156eab5b4,2021-07-05T12:57:17+02:00,BUG: fix no inplace output check for box and s...,4,4,1,src/ufuncs.c,From c05704f69fd0b9ec57fe4b31833736c156eab5b4 ...,True,| File | Lines Added | Lines Removed | Total C...,...,https://api.github.com/repos/pygeos/pygeos/git...,2025-09-18T06:57:08Z,https://api.github.com/repos/pygeos/pygeos,public,388,388,False,cec21ae5f11d9038a7d6cb5ba4fa6f4424eb9b3c,pygeos-pygeos-cec21ae5f11d9038a7d6cb5ba4fa6f44...,diff --git a/src/ufuncs.c b/src/ufuncs.c\ninde...
2,65203a9e763019c42865be1423e267f94ca3f649,2020-11-29T16:03:51-08:00,ENH: Adds reverse function for GEOS >= 3.7 (#2...,136,11,4,CHANGELOG.rst\npygeos/constructive.py\npygeos/...,From 65203a9e763019c42865be1423e267f94ca3f649 ...,True,| File | Lines Added | Lines Removed | Total C...,...,https://api.github.com/repos/pygeos/pygeos/git...,2025-09-18T06:57:08Z,https://api.github.com/repos/pygeos/pygeos,public,388,388,False,3485ffb0db54f055a5379946dcc31984fb8b8853,pygeos-pygeos-3485ffb0db54f055a5379946dcc31984...,diff --git a/CHANGELOG.rst b/CHANGELOG.rst\nin...
3,cec21ae5f11d9038a7d6cb5ba4fa6f4424eb9b3c,2021-06-09T08:41:42+02:00,TST: rename head branch for GEOS from 'master'...,6,5,2,.github/workflows/test-linux.yml\nci/install_g...,From cec21ae5f11d9038a7d6cb5ba4fa6f4424eb9b3c ...,True,| File | Lines Added | Lines Removed | Total C...,...,https://api.github.com/repos/pygeos/pygeos/git...,2025-09-18T06:57:08Z,https://api.github.com/repos/pygeos/pygeos,public,388,388,False,3fa59c7c52519ee1eb10f9ba7e49ced306201e9f,pygeos-pygeos-3fa59c7c52519ee1eb10f9ba7e49ced3...,diff --git a/.github/workflows/test-linux.yml ...
4,ddb440b39718300cefc7fb12ce43ee1b719f8942,2021-11-11T20:39:30+01:00,[Done] dwithin for GEOS 3.10.0 (#417)\n\n,123,5,4,CHANGELOG.rst\npygeos/predicates.py\npygeos/te...,From ddb440b39718300cefc7fb12ce43ee1b719f8942 ...,True,| File | Lines Added | Lines Removed | Total C...,...,https://api.github.com/repos/pygeos/pygeos/git...,2025-09-18T06:57:08Z,https://api.github.com/repos/pygeos/pygeos,public,388,388,False,67fe0ade96c7dbf0bd37633fc62fd13df269e4e4,pygeos-pygeos-67fe0ade96c7dbf0bd37633fc62fd13d...,diff --git a/CHANGELOG.rst b/CHANGELOG.rst\nin...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26312,b68e542a2b2cab4700cb5bb25863efd10bbd0f63,2023-12-11T10:18:33-05:00,Use Mamba Instead of Conda When Running Benchm...,19,7,2,.github/workflows/benchmarks.yml\nasv.conf.json,From b68e542a2b2cab4700cb5bb25863efd10bbd0f63 ...,True,| File | Lines Added | Lines Removed | Total C...,...,https://api.github.com/repos/tardis-sn/tardis/...,2025-10-21T23:37:05Z,https://api.github.com/repos/tardis-sn/tardis,public,225,225,False,799e35ba7f333c15b9ab451dac3a287f4d73f4d5,tardis-sn-tardis-799e35ba7f333c15b9ab451dac3a2...,diff --git a/.github/workflows/benchmarks.yml ...
26313,17b1da429ee99c98fd3ae3f140ab7378be769223,2024-07-12T10:18:45-04:00,Refactor and add more benchmarks for montecarl...,305,378,12,.mailmap\nbenchmarks/benchmark_base.py\nbenchm...,From 17b1da429ee99c98fd3ae3f140ab7378be769223 ...,True,| File | Lines Added | Lines Removed | Tot

In [6]:
LIMIT_PER_REPO = 1


def make_task(row):
    owner, repo = row["repo_name"].split("/")
    t = Task(
        owner=owner,
        repo=repo,
        sha=row["pr_base_sha"],
        # python_version=row['analysis_python_version'],
        # env_payload=json.dumps({"dependencies": row['analysis_final_dependencies'].tolist()}),
        tag="pkg",
        # commit_date=pd.to_datetime(row['commit_date']).to_pydatetime(), # <--- replace with real columns.
    )
    return t


commits_df["task"] = commits_df.apply(make_task, axis=1)
commits_df["container_name"] = commits_df["repo_name"].str.replace("/", "-") + "-" + commits_df["pr_base_sha"] + ":run"

In [7]:
import datetime


def within_3_months(unix_time: float) -> bool:
    one_month_ago = datetime.datetime.now() - datetime.timedelta(days=30)
    return datetime.datetime.fromtimestamp(unix_time) >= one_month_ago


valid_registry = {
    k: v for k, v in registry.registry.items() if len(v.building_data) and within_3_months(v.created_unix)
}
imgs = {k.with_tag("run").get_image_name() for k in valid_registry}
print(len(imgs))

final_tasks = (
    commits_df[commits_df["container_name"].isin(imgs)]
    .dropna(subset=["task"])
    .groupby("repo_name")
    .head(LIMIT_PER_REPO)
    .reset_index(drop=True)["task"]
    .tolist()
)
len(final_tasks)

1014


49

In [8]:
from datasmith.execution.resolution.task_utils import resolve_task

IDX = 12
task = final_tasks[IDX]
# task = Task(owner="pytroll", repo="satpy", sha="0836c021069b7b385c5ff6723779aaf4e66926aa", tag="pkg")
# apache-arrow-nanoarrow-11e73a8c85b45e3d49c8c541b4e1497a649fe03c
# task = Task(owner="apache", repo="arrow-nanoarrow", sha="11e73a8c85b45e3d49c8c541b4e1497a649fe03c", tag="pkg")

analysis, task = resolve_task(task, bypass_cache=False)
print(task)
run_labels = gen_run_labels(task, runid=uuid.uuid4().hex)

21:56:00 INFO     datasmith: agent_build_and_validate: task analysis: python_versions=3.11, final_dependencies=['affinegap==1.12', 'btrees==6.2', 'categorical-distance==1.9', 'cffi==2.0.0', 'datetime-distance==0.1.3', 'dedupe==3.0.3', 'dedupe-levenshtein-search==1.4.5', 'dedupe-variable-datetime==2.0.0', 'doublemetaphone==1.2', 'future==1.0.0', 'haversine==2.9.0', 'highered==0.2.1', 'joblib==1.5.2', 'numpy==2.3.4', 'persistent==6.3', 'pycparser==2.23', 'pyhacrf-datamade==0.2.8', 'pylbfgs==0.2.0.16', 'python-dateutil==2.9.0.post0', 'scikit-learn==1.7.2', 'scipy==1.16.3', 'setuptools==80.9.0', 'simplecosine==1.2', 'six==1.17.0', 'threadpoolctl==3.6.0', 'typing-extensions==4.15.0', 'zope-deferredimport==6.0', 'zope-index==8.0', 'zope-interface==8.0.1', 'zope-proxy==7.0']


Task(owner='dedupeio', repo='dedupe', sha='4e44a5fcf8c982bdb1c981bb425772c88c2b380c', commit_date=0.0, env_payload='{"dependencies": ["affinegap==1.12", "btrees==6.2", "categorical-distance==1.9", "cffi==2.0.0", "datetime-distance==0.1.3", "dedupe==3.0.3", "dedupe-levenshtein-search==1.4.5", "dedupe-variable-datetime==2.0.0", "doublemetaphone==1.2", "future==1.0.0", "haversine==2.9.0", "highered==0.2.1", "joblib==1.5.2", "numpy==2.3.4", "persistent==6.3", "pycparser==2.23", "pyhacrf-datamade==0.2.8", "pylbfgs==0.2.0.16", "python-dateutil==2.9.0.post0", "scikit-learn==1.7.2", "scipy==1.16.3", "setuptools==80.9.0", "simplecosine==1.2", "six==1.17.0", "threadpoolctl==3.6.0", "typing-extensions==4.15.0", "zope-deferredimport==6.0", "zope-index==8.0", "zope-interface==8.0.1", "zope-proxy==7.0"]}', python_version='3.11', tag='pkg', benchmarks='')


## Build + Validate with your script

In [9]:
similar_ctx = registry.get_similar(task)
len(similar_ctx)

14

In [10]:
# Create a DockerContext from your building_data
# building_data = Path("scratch/manual_docker_build_pkg.sh").read_text()
# ctx = DockerContext(building_data=building_data)
SIMILAR_IDX = 0
ctx = DockerContext(building_data=similar_ctx[SIMILAR_IDX][1].building_data)


# Build & validate the 'run' image; validator will run profile then tests
res = validator.build_and_validate(
    task=task.with_tag("run"),
    context=ctx,
    run_labels=run_labels,
    build_once_fn=build_once_with_context,
)

print("ok:", res.ok, "rc:", res.rc)
print("duration_s:", res.duration_s)

print(res.stdout_tail)

21:56:06 INFO     datasmith.docker.validation: build_and_validate[dedupeio-dedupe-4e44a5fcf8c982bdb1c981bb425772c88c2b380c:run]: building image
21:56:06 INFO     datasmith.docker.context: Docker image 'dedupeio-dedupe-4e44a5fcf8c982bdb1c981bb425772c88c2b380c:run' found locally (skip build).
21:56:06 INFO     datasmith.agents.build: build_once_with_context: result ok=True rc=0 duration=0.0s (stderr_tail_len=0, stdout_tail_len=0)
21:56:06 INFO     datasmith.docker.validation: build_and_validate[dedupeio-dedupe-4e44a5fcf8c982bdb1c981bb425772c88c2b380c:run]: build ok; verifying profile+tests before recording attempt


21:56:18 INFO     datasmith.docker.validation: build_and_validate[dedupeio-dedupe-4e44a5fcf8c982bdb1c981bb425772c88c2b380c:run]: profile:done ok=True duration=12.1s
21:56:27 WARNING  datasmith.docker.validation: build_and_validate[dedupeio-dedupe-4e44a5fcf8c982bdb1c981bb425772c88c2b380c:run]: test validation failed, but ignoring test errors


ok: True rc: 0
duration_s: 0.0043087005615234375
=== BUILD ===
ok: True rc: 0
duration_s: 0.00
(no additional stdout)

=== PROFILE VALIDATION ===
ok: True rc: 0
duration_s: 12.03

--- Benchmarks ---
canonical.Canonical.peakmem_run
canonical.Canonical.time_run
canonical.Canonical.track_precision
canonical.Canonical.track_recall
canonical_gazetteer.Gazetteer.peakmem_run
canonical_gazetteer.Gazetteer.time_run
canonical_gazetteer.Gazetteer.track_precision
canonical_gazetteer.Gazetteer.track_recall
canonical_matching.Matching.peakmem_run
canonical_matching.Matching.time_run
canonical_matching.Matching.track_precision
canonical_matching.Matching.track_recall


=== TEST VALIDATION ===
(no test stdout; showing condensed errors)
+ cd /workspace/repo
+ set +ux
+ '[' 0 -gt 0 ']'
+ '[' -n 4e44a5fcf8c982bdb1c981bb425772c88c2b380c ']'
+ FORMULACODE_BASE_COMMIT=4e44a5fcf8c982bdb1c981bb425772c88c2b380c
+ reset_repo_state 4e44a5fcf8c982bdb1c981bb425772c88c2b380c
+ local COMMIT_SHA=4e44a5fcf8c982bdb1c98

In [11]:
print(res.benchmarks)

canonical.Canonical.peakmem_run
canonical.Canonical.time_run
canonical.Canonical.track_precision
canonical.Canonical.track_recall
canonical_gazetteer.Gazetteer.peakmem_run
canonical_gazetteer.Gazetteer.time_run
canonical_gazetteer.Gazetteer.track_precision
canonical_gazetteer.Gazetteer.track_recall
canonical_matching.Matching.peakmem_run
canonical_matching.Matching.time_run
canonical_matching.Matching.track_precision
canonical_matching.Matching.track_recall



In [12]:
final_task = task.with_tag("final").with_benchmarks(res.benchmarks)

res2 = build_once_with_context(
    client=client,
    task=final_task,
    context=ctx,
    repo_url=f"https://www.github.com/{final_task.owner}/{final_task.repo}",
    sha=final_task.sha or "",
    force=True,
    timeout_s=1024,
    tail_chars=10_000,
    run_labels={},
)

21:56:46 INFO     datasmith.agents.build: build_once_with_context: injecting benchmarks into build args
21:56:46 INFO     datasmith.docker.context: Force rebuild requested. Removing 'dedupeio-dedupe-4e44a5fcf8c982bdb1c981bb425772c88c2b380c:final'.


21:56:46 INFO     datasmith.docker.context: $ docker build -t dedupeio-dedupe-4e44a5fcf8c982bdb1c981bb425772c88c2b380c:final . --build-arg REPO_URL='https://www.github.com/dedupeio/dedupe' --build-arg COMMIT_SHA='4e44a5fcf8c982bdb1c981bb425772c88c2b380c' --build-arg ENV_PAYLOAD='{"dependencies": ["affinegap==1.12", "btrees==6.2", "categorical-distance==1.9", "cffi==2.0.0", "datetime-distance==0.1.3", "dedupe==3.0.3", "dedupe-levenshtein-search==1.4.5", "dedupe-variable-datetime==2.0.0", "doublemetaphone==1.2", "future==1.0.0", "haversine==2.9.0", "highered==0.2.1", "joblib==1.5.2", "numpy==2.3.4", "persistent==6.3", "pycparser==2.23", "pyhacrf-datamade==0.2.8", "pylbfgs==0.2.0.16", "python-dateutil==2.9.0.post0", "scikit-learn==1.7.2", "scipy==1.16.3", "setuptools==80.9.0", "simplecosine==1.2", "six==1.17.0", "threadpoolctl==3.6.0", "typing-extensions==4.15.0", "zope-deferredimport==6.0", "zope-index==8.0", "zope-interface==8.0.1", "zope-proxy==7.0"]}' --build-arg PY_VERSION='3.11' --b

In [28]:
res2

BuildResult(ok=True, image_name='dedupeio-dedupe-4e44a5fcf8c982bdb1c981bb425772c88c2b380c:final', image_id='sha256:fdc497b65d545d11494d43e85aa97d5561fd3a2f4da134afdf547e06c2dd7c70', rc=0, duration_s=0.004135608673095703, stderr_tail='', stdout_tail='', failure_stage=None, benchmarks='')

## Register on success

In [ ]:
if res.ok:
    _handle_success(
        ctx=ctx,
        task=task,
        context_registry=registry,
        client=client,
        args=argparse.Namespace(
            context_registry=CONTEXT_REGISTRY_PATH,
            output_dir=OUTPUT_DIR,
            push_to_ecr=False,  # Do this manually at the end. Slow.
        ),
        build_result=res,
    )

22:04:25 INFO     datasmith.docker.context: Context registry saved to scratch/merged_context_registry_2025-10-08T09:22:23.904388.json
22:04:25 INFO     datasmith.agents.build: Saved DockerContext pickle: scverse-spatialdata-cab8353e7549a04b2a2538990e0afc2935e54f3f-final.pkl


## Optional: cleanup containers for this run

In [ ]:
# Remove labeled containers; images are left intact
run_id = run_labels.get("datasmith.run", "unknown")
remove_containers_by_label(client, run_id)
print("Cleaned up containers for run_id:", run_id)